In [ ]:
#Importing libraries
import pandas as pd
import nltk
nltk.download('all')
from sklearn.metrics import classification_report

[nltk_data] Downloading collection 'all'
[nltk_data]    | 
[nltk_data]    | Downloading package abc to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/abc.zip.
[nltk_data]    | Downloading package alpino to /root/nltk_data...
[nltk_data]    |   Unzipping corpora/alpino.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping taggers/averaged_perceptron_tagger.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_eng to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_eng.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_ru to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |   Unzipping
[nltk_data]    |       taggers/averaged_perceptron_tagger_ru.zip.
[nltk_data]    | Downloading package averaged_perceptron_tagger_rus to
[nltk_data]    |     /root/nltk_data...
[nltk_data]    |  

In [ ]:
train_df = pd.read_csv('train.csv.zip')
train_df.head()

,qtype,Question,Answer
0,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,LCMV infections can occur after exposure to fr...
1,symptoms,What are the symptoms of Lymphocytic Choriomen...,LCMV is most commonly recognized as causing ne...
2,susceptibility,Who is at risk for Lymphocytic Choriomeningiti...,Individuals of all ages who come into contact ...
3,exams and tests,How to diagnose Lymphocytic Choriomeningitis (...,"During the first phase of the disease, the mos..."
4,treatment,What are the treatments for Lymphocytic Chorio...,"Aseptic meningitis, encephalitis, or meningoen..."


In [ ]:
print("Number of medical answers: ", len(train_df))
print('number of patient questions: ', len(train_df))

Number of medical answers:  16407
number of patient questions:  16407


## Data Preparation

In [ ]:
#Check rows with nulls
train_df.isnull().sum().sort_values(ascending=False)

,0
qtype,0
Question,0
Answer,0


In [ ]:
#Remove rows with nulls, Answer
train_df = train_df[train_df['Answer'].notnull()]
train_df.info()

# Remove rows with nulls, Question
train_df = train_df[train_df['Question'].notnull()]
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16407 entries, 0 to 16406
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   qtype     16407 non-null  object
 1   Question  16407 non-null  object
 2   Answer    16407 non-null  object
dtypes: object(3)
memory usage: 384.7+ KB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16407 entries, 0 to 16406
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   qtype     16407 non-null  object
 1   Question  16407 non-null  object
 2   Answer    16407 non-null  object
dtypes: object(3)
memory usage: 384.7+ KB


In [ ]:
#Convert Answer into lowercase
train_df['Answer'] = train_df['Answer'].str.lower()
train_df.head()

# Convert Question into lowercase
train_df['Question'] = train_df['Question'].str.lower()
train_df.head()

,qtype,Question,Answer
0,susceptibility,who is at risk for lymphocytic choriomeningiti...,lcmv infections can occur after exposure to fr...
1,symptoms,what are the symptoms of lymphocytic choriomen...,lcmv is most commonly recognized as causing ne...
2,susceptibility,who is at risk for lymphocytic choriomeningiti...,individuals of all ages who come into contact ...
3,exams and tests,how to diagnose lymphocytic choriomeningitis (...,"during the first phase of the disease, the mos..."
4,treatment,what are the treatments for lymphocytic chorio...,"aseptic meningitis, encephalitis, or meningoen..."


In [ ]:
#Remove stop words
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

stop_words = set(stopwords.words('english'))

def remove_stopwords(text):
  tokens = word_tokenize(text)
  filtered_tokens = [word for word in tokens if word.casefold() not in stop_words]
  return ' '.join(filtered_tokens)

#Answer
train_df['Answer_no_stopwords'] = train_df['Answer'].apply(remove_stopwords)
train_df[['Answer','Answer_no_stopwords']].head()


,Answer,Answer_no_stopwords
0,lcmv infections can occur after exposure to fr...,"lcmv infections occur exposure fresh urine , d..."
1,lcmv is most commonly recognized as causing ne...,lcmv commonly recognized causing neurological ...
2,individuals of all ages who come into contact ...,"individuals ages come contact urine , feces , ..."
3,"during the first phase of the disease, the mos...","first phase disease , common laboratory abnorm..."
4,"aseptic meningitis, encephalitis, or meningoen...","aseptic meningitis , encephalitis , meningoenc..."


In [ ]:
Answers=train_df['Answer_no_stopwords']
labels = train_df['qtype']
answer_data_with_labels = list(zip(Answers, labels))
print('answer_data_with_labels[0]: ', answer_data_with_labels[0])

answer_data_with_labels[0]:  ('lcmv infections occur exposure fresh urine , droppings , saliva , nesting materials infected rodents . transmission may also occur materials directly introduced broken skin , nose , eyes , mouth , presumably , via bite infected rodent . person-to-person transmission reported , exception vertical transmission infected mother fetus , rarely , organ transplantation .', 'susceptibility')


## Split the Dataset

In [ ]:
#split the dataset
from sklearn.model_selection import train_test_split
#answer
answer_train_set, answer_test_set = train_test_split(answer_data_with_labels, test_size=0.2, random_state=7)
print('Number of training medical answer: ', len(answer_train_set))
print('train_set[0]: ',answer_train_set[0])
print(' Number of testing medical answer: ', len(answer_test_set))


Number of training medical answer:  13125
train_set[0]:  ('noonan-like syndrome loose anagen hair characterized facial features suggestive noonan syndrome ( macrocephaly , high forehead , wide-set eyes hypertelorism , palpebral ptosis , low-set posteriorly rotated ears ) along hair resembles loose anagen hair syndrome ( pluckable , sparse , thin slow-growing ) . features include frequent congenital heart defects , distinctive skin features ( darkly pigmented skin eczema ichthyosis ) , short stature may associated growth hormone deficiency , developmental delays . condition caused mutations shoc2 gene . follows autosomal dominant pattern inheritance .', 'information')
 Number of testing medical answer:  3282


# **unigram feature sets (part of project)**



In [ ]:
# Unigrams for Doctor language (Answer column)
def unigram_features_answer(data) -> [(dict,str)]:
  med_answers = [d[0] for d in data]
  med_qtype = [d[1] for d in data]
  features_dicts = [{word: True for word in nltk.word_tokenize(str(med_answer))} for med_answer in med_answers]
  return list(zip(features_dicts, med_qtype))

# Print example
answer_unigram_set = unigram_features_answer(answer_train_set)
print(answer_unigram_set[10])

({'weakened': True, 'immune': True, 'system': True, 'may': True, 'increase': True, 'risk': True, 'developing': True, 'primary': True, 'cns': True, 'lymphoma': True, '.': True, 'anything': True, 'increases': True, 'chance': True, 'getting': True, 'disease': True, 'called': True, 'factor': True, 'mean': True, 'get': True, 'cancer': True, ';': True, 'factors': True, 'doesnt': True, 'talk': True, 'doctor': True, 'think': True, 'occur': True, 'patients': True, 'acquired': True, 'immunodeficiency': True, 'syndrome': True, '(': True, 'aids': True, ')': True, 'disorders': True, 'kidney': True, 'transplant': True, 'information': True, ',': True, 'see': True, 'pdq': True, 'summary': True, 'aids-related': True, 'treatment': True}, 'susceptibility')


# Training the model for unigram

In [ ]:
answer_feature_sets = unigram_features_answer(answer_train_set)
classifier = nltk.NaiveBayesClassifier.train(answer_feature_sets)

In [ ]:
# Prediction
test_feature_sets = unigram_features_answer(answer_test_set)
test_dict = test_feature_sets[0][0]
print('Real Value: ', test_feature_sets[0][1])
prediction = classifier.classify(test_dict)
print('Predicted Value for answer: ', prediction)

# Evaluation
def evaluate_classifier(classifier, test_data) -> float:
    test_feature_sets = unigram_features_answer(test_data)
    return nltk.classify.accuracy(classifier, test_feature_sets)

acc = evaluate_classifier(classifier, answer_test_set)
print('Accuracy: ', acc)

Real Value:  information
Predicted Value for answer:  support groups
Accuracy:  0.17093235831809872


# Classification Report for unigram

In [ ]:
num_test_labels = len(answer_test_set)
print('num_test_labels: ', num_test_labels)

true_labels = []
for i in range(num_test_labels):
    true_labels.append(test_feature_sets[i][1])
print('true_labels: ', true_labels)

predicted_labels = []
for i in range(num_test_labels):
    prediction_q = classifier.classify(test_feature_sets[i][0])
    predicted_labels.append(prediction_q)
print('predicted_labels: ', predicted_labels)

print(classification_report(true_labels, predicted_labels))

num_test_labels:  3282
true_labels:  ['information', 'inheritance', 'susceptibility', 'information', 'information', 'treatment', 'treatment', 'information', 'inheritance', 'genetic changes', 'information', 'information', 'treatment', 'genetic changes', 'frequency', 'information', 'symptoms', 'symptoms', 'information', 'information', 'causes', 'symptoms', 'information', 'information', 'treatment', 'information', 'considerations', 'symptoms', 'prevention', 'susceptibility', 'frequency', 'symptoms', 'inheritance', 'genetic changes', 'symptoms', 'genetic changes', 'information', 'symptoms', 'information', 'symptoms', 'symptoms', 'symptoms', 'information', 'information', 'inheritance', 'information', 'symptoms', 'treatment', 'treatment', 'susceptibility', 'causes', 'information', 'susceptibility', 'prevention', 'research', 'information', 'exams and tests', 'genetic changes', 'treatment', 'information', 'frequency', 'symptoms', 'genetic changes', 'symptoms', 'frequency', 'inheritance', 'gene

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

# ------------------------------------------------------

## **bigram feature sets (part of project)**

In [ ]:
# Bigrams for Doctor language (Answer column)
def bigram_features_answer(data) -> [(dict,str)]:
  med_answers = [d[0] for d in data]
  med_qtype = [d[1] for d in data]
  features_dicts = [{word: True for word in nltk.bigrams(nltk.word_tokenize(str(med_answer)))} for med_answer in med_answers]
  return list(zip(features_dicts, med_qtype))

# Print example of doctor bigrams
answer_feature_set = bigram_features_answer(answer_train_set)
print(answer_feature_set[10])

({('weakened', 'immune'): True, ('immune', 'system'): True, ('system', 'may'): True, ('may', 'increase'): True, ('increase', 'risk'): True, ('risk', 'developing'): True, ('developing', 'primary'): True, ('primary', 'cns'): True, ('cns', 'lymphoma'): True, ('lymphoma', '.'): True, ('.', 'anything'): True, ('anything', 'increases'): True, ('increases', 'chance'): True, ('chance', 'getting'): True, ('getting', 'disease'): True, ('disease', 'called'): True, ('called', 'risk'): True, ('risk', 'factor'): True, ('factor', '.'): True, ('.', 'risk'): True, ('factor', 'mean'): True, ('mean', 'get'): True, ('get', 'cancer'): True, ('cancer', ';'): True, (';', 'risk'): True, ('risk', 'factors'): True, ('factors', 'doesnt'): True, ('doesnt', 'mean'): True, ('cancer', '.'): True, ('.', 'talk'): True, ('talk', 'doctor'): True, ('doctor', 'think'): True, ('think', 'may'): True, ('may', 'risk'): True, ('risk', '.'): True, ('.', 'primary'): True, ('lymphoma', 'may'): True, ('may', 'occur'): True, ('occu

# Training the model for bigram

In [ ]:
answer_bigram_feature_sets = bigram_features_answer(answer_train_set)
classifier_bigram = nltk.NaiveBayesClassifier.train(answer_bigram_feature_sets)

# Prediction and Evaluation for bigram

In [ ]:
test_bigram_feature_sets = bigram_features_answer(answer_test_set)
test_dict = test_bigram_feature_sets[0][0]
print('Real Value: ', test_bigram_feature_sets[0][1])
prediction = classifier_bigram.classify(test_dict)
print('Predicted Value: ', prediction)

# Evaluation
acc_bigram = nltk.classify.accuracy(classifier_bigram, test_bigram_feature_sets)
print('Accuracy: ', acc_bigram)

Real Value:  information
Predicted Value:  support groups
Accuracy:  0.11608775137111517


# Classification Report for bigram

In [ ]:
num_test_labels = len(answer_test_set)
true_labels = [test_bigram_feature_sets[i][1] for i in range(num_test_labels)]
predicted_labels = [classifier_bigram.classify(test_bigram_feature_sets[i][0]) for i in range(num_test_labels)]
print(classification_report(true_labels, predicted_labels))

                 precision    recall  f1-score   support

         causes       0.00      0.00      0.00       140
  complications       0.00      0.00      0.00         8
 considerations       0.00      0.00      0.00        46
exams and tests       0.00      0.00      0.00       124
      frequency       1.00      0.02      0.03       252
genetic changes       0.00      0.00      0.00       213
    information       0.00      0.00      0.00       883
    inheritance       1.00      0.34      0.50       280
        outlook       0.00      0.00      0.00        61
     prevention       0.00      0.00      0.00        51
       research       0.00      0.00      0.00        74
         stages       1.00      0.08      0.14        13
 support groups       0.00      0.00      0.00         0
 susceptibility       0.00      0.00      0.00        74
       symptoms       1.00      0.40      0.58       588
      treatment       1.00      0.09      0.17       475

       accuracy              

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

# ---------------------------------------------------

## **trigram feature sets (part of project)**

In [ ]:
# Trigrams for Doctor language (Answer column)
def trigram_features_answer(data) -> [(dict,str)]:
  med_answers = [d[0] for d in data]
  med_qtype = [d[1] for d in data]
  features_dicts = [{word: True for word in nltk.trigrams(nltk.word_tokenize(str(med_answer)))} for med_answer in med_answers]
  return list(zip(features_dicts, med_qtype))

# Print example
answer_trigram_set = trigram_features_answer(answer_train_set)
print(answer_trigram_set[10])

({('weakened', 'immune', 'system'): True, ('immune', 'system', 'may'): True, ('system', 'may', 'increase'): True, ('may', 'increase', 'risk'): True, ('increase', 'risk', 'developing'): True, ('risk', 'developing', 'primary'): True, ('developing', 'primary', 'cns'): True, ('primary', 'cns', 'lymphoma'): True, ('cns', 'lymphoma', '.'): True, ('lymphoma', '.', 'anything'): True, ('.', 'anything', 'increases'): True, ('anything', 'increases', 'chance'): True, ('increases', 'chance', 'getting'): True, ('chance', 'getting', 'disease'): True, ('getting', 'disease', 'called'): True, ('disease', 'called', 'risk'): True, ('called', 'risk', 'factor'): True, ('risk', 'factor', '.'): True, ('factor', '.', 'risk'): True, ('.', 'risk', 'factor'): True, ('risk', 'factor', 'mean'): True, ('factor', 'mean', 'get'): True, ('mean', 'get', 'cancer'): True, ('get', 'cancer', ';'): True, ('cancer', ';', 'risk'): True, (';', 'risk', 'factors'): True, ('risk', 'factors', 'doesnt'): True, ('factors', 'doesnt', 

# Training the model for trigram

In [ ]:
# Fixed: train the trigram classifier on trigram features (it was trained on unigram features)
answer_trigram_feature_sets = trigram_features_answer(answer_train_set)
classifier_trigram = nltk.NaiveBayesClassifier.train(answer_trigram_feature_sets)

# Prediction and Evaluation for trigram

In [ ]:
# Prediction
test_trigram_feature_sets = trigram_features_answer(answer_test_set)
test_dict = test_trigram_feature_sets[0][0]
print('Real Value: ', test_trigram_feature_sets[0][1])
prediction = classifier_trigram.classify(test_dict)
print('Predicted Value: ', prediction)

# Evaluation
acc_trigram = nltk.classify.accuracy(classifier_trigram, test_trigram_feature_sets)
print('Accuracy: ', acc_trigram)

Classification Report for trigram

In [ ]:
# Classification Report
num_test_labels = len(answer_test_set)
true_labels = [test_trigram_feature_sets[i][1] for i in range(num_test_labels)]
predicted_labels = [classifier_trigram.classify(test_trigram_feature_sets[i][0]) for i in range(num_test_labels)]
print(classification_report(true_labels, predicted_labels))